In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [35]:
df = pd.read_csv("quikr_car.csv")
df.head()

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"80,000","45,000 kms",Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,"4,25,000",40 kms,Diesel
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,Ask For Price,"22,000 kms",Petrol
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"3,25,000","28,000 kms",Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"5,75,000","36,000 kms",Diesel


In [40]:

df = df[df['year'].astype(str).str.isnumeric()]
df['year'] = df['year'].astype(int)


df = df.dropna()


df['kms_driven'] = df['kms_driven'].astype(str)
df['kms_driven'] = df['kms_driven'].str.replace(',', '')
df['kms_driven'] = df['kms_driven'].str.replace(' kms', '', regex=False)
df['kms_driven'] = pd.to_numeric(df['kms_driven'], errors='coerce')

df = df.dropna()
df['kms_driven'] = df['kms_driven'].astype(int)

In [41]:
X = df[['name', 'company', 'year', 'kms_driven', 'fuel_type']]
y = df['Price']

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [43]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ohe = OneHotEncoder(handle_unknown='ignore')

transformer = ColumnTransformer(
    [('encoder', ohe, ['name', 'company', 'fuel_type'])],
    remainder='passthrough'
)

In [44]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

pipe = Pipeline([
    ('transformer', transformer),
    ('model', LinearRegression())
])

pipe.fit(X_train, y_train)

Pipeline(steps=[('transformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['name', 'company',
                                                   'fuel_type'])])),
                ('model', LinearRegression())])

In [45]:
y_pred = pipe.predict(X_test)

In [46]:
from sklearn.metrics import r2_score

score = r2_score(y_test, y_pred)

print("R2 Score:", score)

R2 Score: 0.1899542870155545


In [47]:
sample = pd.DataFrame({
    'name': ['Hyundai Santro Xing XO eRLX Euro III'],
    'company': ['Hyundai'],
    'year': [2007],
    'kms_driven': [45000],
    'fuel_type': ['Petrol']
})

price = pipe.predict(sample)

print("Predicted Price: ₹", int(price[0]))

Predicted Price: ₹ 89089
